# Train the CNN guidance-map (Colab / GPU) — memory-frugal

Consumes `guidance_dataset*.npz` shards from `python -m ml_planner.build_dataset`
(hard maps, GRID_RES=384).

**Fixes the RAM overflow:** streams **one shard at a time** (never loads the
whole dataset), uses a **small 2-level U-Net** (base=16) and **mixed precision**.

**Hard I/O contract (must match `ml_planner/guidance.py`):** input `channels`
`(1,4,384,384)` float32, output `cost_to_go` `(1,1,384,384)` float32,
opset>=11, names exactly `channels`/`cost_to_go`. The net predicts a **residual
over Euclid** (output adds the dist-to-goal channel). torch is used ONLY here.

Set **Runtime -> Change runtime type -> GPU**.


In [ ]:
# 1. List shard paths (do NOT load them all — we stream per shard).
import glob, os, random, numpy as np, torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
try:
    from google.colab import files
    up = files.upload()                 # select ALL guidance_dataset*.npz shards
    PATHS = sorted(up.keys())
except Exception:
    PATHS = sorted(glob.glob('guidance_dataset*.npz'))
assert PATHS, 'no guidance_dataset*.npz found'
GRID = 384
print('shards:', len(PATHS))


In [ ]:
# 2. Per-shard loader (loads ONE shard, normalizes labels by crop diagonal).
def load_shard(p):
    d = np.load(p)
    ch = d['channels'].astype('float32')
    af = d['affine']
    diag = (np.sqrt(2.0) * (af[:, 3] / af[:, 2])).astype('float32')
    lan = d['label'].astype('float32') / diag[:, None, None]
    ms = d['mask'].astype('float32')
    return ch, lan, ms

# Hold out the last shard for validation; stream the rest for training.
if len(PATHS) >= 2:
    TRAIN_PATHS, VAL = PATHS[:-1], [load_shard(PATHS[-1])]
else:
    ch, lan, ms = load_shard(PATHS[0]); nval = max(1, len(ch)//5)
    VAL = [(ch[:nval], lan[:nval], ms[:nval])]; TRAIN_PATHS = PATHS
print('train shards', len(TRAIN_PATHS), '| val samples', sum(len(v[0]) for v in VAL))


In [ ]:
# 3. Small 2-level U-Net; output adds dist-to-goal channel => residual-over-Euclid.
class DoubleConv(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
            nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, cin=4, base=16):
        super().__init__()
        self.d1 = DoubleConv(cin, base);      self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(base, base*2);   self.p2 = nn.MaxPool2d(2)
        self.mid = DoubleConv(base*2, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2); self.c2 = DoubleConv(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, stride=2);   self.c1 = DoubleConv(base*2, base)
        self.head = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        x1 = self.d1(x); x2 = self.d2(self.p1(x1)); m = self.mid(self.p2(x2))
        y = self.c2(torch.cat([self.u2(m), x2], 1))
        y = self.c1(torch.cat([self.u1(y), x1], 1))
        return self.head(y) + x[:, 2:3]

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = dev == 'cuda'
model = UNet(base=16).to(dev)
print('device', dev, '| params', sum(p.numel() for p in model.parameters()), '| amp', use_amp)


In [ ]:
# 4. Stream-train with masked MSE + mixed precision + early stopping.
EPOCHS, bs, PATIENCE = 60, 8, 10
opt = torch.optim.Adam(model.parameters(), 2e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = GradScaler(enabled=use_amp)

def masked_mse(pred, y, m):
    m = m > 0
    return (((pred - y) ** 2) * m).sum() / m.sum().clamp(min=1)

def val_loss():
    model.eval(); tot = cnt = 0
    with torch.no_grad():
        for ch, lan, ms in VAL:
            for i in range(0, len(ch), bs):
                xb = torch.tensor(ch[i:i+bs]).to(dev); yb = torch.tensor(lan[i:i+bs]).to(dev); mb = torch.tensor(ms[i:i+bs]).to(dev)
                with autocast(enabled=use_amp):
                    l = masked_mse(model(xb)[:, 0], yb, mb)
                tot += float(l) * len(xb); cnt += len(xb)
    return tot / max(1, cnt)

best, best_state, since = 1e9, None, 0
for epoch in range(EPOCHS):
    model.train(); order = list(TRAIN_PATHS); random.shuffle(order)
    for p in order:
        ch, lan, ms = load_shard(p); idx = np.random.permutation(len(ch))
        for i in range(0, len(idx), bs):
            j = idx[i:i+bs]
            xb = torch.tensor(ch[j]).to(dev); yb = torch.tensor(lan[j]).to(dev); mb = torch.tensor(ms[j]).to(dev)
            opt.zero_grad()
            with autocast(enabled=use_amp):
                loss = masked_mse(model(xb)[:, 0], yb, mb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        del ch, lan, ms
    sched.step(); vl = val_loss()
    if vl < best - 1e-9:
        best, best_state, since = vl, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
    else:
        since += 1
    if epoch % 5 == 0 or since == 0:
        print(f'epoch {epoch:3d}  val masked-MSE {vl:.6f}  best {best:.6f}')
    if since >= PATIENCE:
        print('early stop at epoch', epoch); break
print('best val masked-MSE', best)
model.load_state_dict(best_state)


In [ ]:
# 5. Export a SINGLE self-contained ONNX (channels -> cost_to_go, 384x384) and download.
model.eval().cpu()
dummy = torch.zeros(1, 4, 384, 384)
torch.onnx.export(
    model, dummy, 'guidance.onnx',
    input_names=['channels'], output_names=['cost_to_go'], opset_version=13,
    dynamic_axes={'channels': {0: 'batch'}, 'cost_to_go': {0: 'batch'}})
import onnx
onnx.save_model(onnx.load('guidance.onnx'), 'guidance.onnx', save_as_external_data=False)
print('exported single-file guidance.onnx')
try:
    from google.colab import files
    files.download('guidance.onnx')
except Exception:
    pass
# Drop guidance.onnx into ml_planner/models/ ; secondary='guidance' then activates.
